# 5 · The PLS projection

Notebook 2 re-expressed the 1536-d latent θ in the **PCA** basis: an unsupervised
rotation that asks *where does the latent vary most?* This notebook covers the other
projection the package ships — **PLS** — which asks a different question:

> *Which directions in the latent covary most with stellar age?*

PLS (partial least squares) is **supervised**: it uses `log10(age/Myr)` to choose its
directions. That makes it sharper than PCA at surfacing age structure, and it also makes
it a fundamentally different object with caveats PCA does not have. Both are covered
below.

What you'll get: the exact scaling and centering applied before projection, the
projection matrix itself, how to keep any number of components, and when not to trust
the result.

In [ ]:
import numpy as np
import encotess
from encotess import assets

pls = encotess.GlobalPLS.from_npz(assets.pls_projection_path())

print('components available :', pls.n_components)
print('fit on               :', pls.n_fit_stars, 'labelled stars')
print('target               :', pls.target)
print('star aggregation     :', pls.aggregation)

## The transform, in full

The projection is **affine** — two steps, no hidden state:

$$
X_{\text{std}} = \frac{X - \mathrm{x\_mean}}{\mathrm{x\_std}},
\qquad
T = X_{\text{std}} \; R_{:, :k}
$$

**Step 1 — centre *and* scale, per latent dimension.** `x_mean` and `x_std` are the
per-feature mean and standard deviation of the 1536 latent dimensions, measured on the
fit stars. This is sklearn's `PLSRegression(scale=True)`: unlike plain centering, it
also divides by the standard deviation, so every latent dimension enters the fit with
equal weight regardless of its native scale.

**Step 2 — project.** `x_rotations` is the $(1536, 16)$ matrix of PLS weights. Keeping
`dim=k` columns gives $k$ component scores.

That is the whole thing. The arrays:

In [ ]:
print(f'x_mean      {str(pls.x_mean.shape):12s}  first 3: {pls.x_mean[:3].round(4)}')
print(f'x_std       {str(pls.x_std.shape):12s}  first 3: {pls.x_std[:3].round(4)}')
print(f'x_rotations {str(pls.x_rotations.shape):12s}  column norms: '
      f'{np.linalg.norm(pls.x_rotations, axis=0)[:4].round(4)} ...')

### How this differs from the PCA basis

| | `project_pca` | `project_pls` |
|---|---|---|
| Uses the target? | No — unsupervised | **Yes**, `log10(age/Myr)` |
| Maximises | variance of the projection | **covariance** with age |
| Pre-projection | standardize, then subtract `pca_mean` | standardize (centre **and** scale) |
| Matrix | orthonormal, full-rank 1536 | $(1536, 16)$, **not orthonormal** |
| Invertible? | Yes — a lossless rotation | **No** — information is discarded |
| Explained variance? | Yes, per component | **No such quantity** |

The non-orthonormality is easy to check, and it is the reason there is no inverse and no
variance ratio to report:

In [ ]:
R = pls.x_rotations
G = R.T @ R                      # would be the identity for an orthonormal basis
off = G - np.diag(np.diag(G))
print(f'max |off-diagonal| of RᵀR : {np.abs(off).max():.4f}   (0 if orthonormal)')
print(f'diagonal of RᵀR, first 4  : {np.diag(G)[:4].round(4)}   (1 if orthonormal)')

## Rows are stars, not light curves

The released PLS view has one row per **star**, while latents are produced per **light
curve**. A star's sector latents are combined with an **element-wise maximum** before
projection — that is what `aggregation = 'max'` in the artifact means, and matching it is
required to reproduce the released scores.

`aggregate_by_star` does this (and also offers `'mean'` / `'median'` if you want a
different summary for your own work).

In [ ]:
rng = np.random.default_rng(0)
# Three stars: 4, 1 and 2 sector light curves respectively.
gaia_ids = np.array(['star_A'] * 4 + ['star_B'] + ['star_C'] * 2)
latents = rng.normal(size=(7, 1536))

uniq, Z = encotess.aggregate_by_star(latents, gaia_ids, how='max')
print('light curves in :', latents.shape)
print('stars out       :', Z.shape, '->', list(uniq))
print('is element-wise max of star_A rows:',
      np.allclose(Z[0], latents[:4].max(axis=0)))

## Projecting, and keeping any number of components

PLS components are **nested**: the first $k$ columns of the 16-component fit are
*exactly* what a $k$-component fit would produce (they agree to ~1e-16). So a single
bundled $(1536, 16)$ matrix serves every $k \le 16$ — `dim` just slices it, no refitting.

This is unlike PCA only in flavour, not in mechanics: both let you truncate. The
difference is that PLS components are ordered by **age covariance**, so component 1 is
the single most age-informative direction available, not the highest-variance one.

In [ ]:
T_full = pls.transform(Z)              # all 16
T3     = pls.transform(Z, dim=3)       # top 3
print('full:', T_full.shape, ' dim=3:', T3.shape)
print('slicing == truncating:', np.array_equal(T3, T_full[:, :3]))

# Equivalently, straight off the encoder:
enc = encotess.load_encoder(device='cpu')
print('via Encoder.project_pls:', enc.project_pls(Z[0], dim=4).round(3))

### Reproducing the released scores

The package also ships the precomputed PLS view for every released star
(`encodings_pls16_star.npz`). Applying the projection to the `latent_max`-aggregated
latents reproduces it — a direct check that the pipeline above is the one that built the
released file.

The cell below needs the full 1536-d latents, which are a ~400 MB download
(`assets.download_latents_pca()` gives the PCA form, losslessly invertible back to the
latent). It is written out rather than run so the notebook stays offline; the measured
agreement when it *is* run is a median residual of **3.3e-6** of a component standard
deviation, with 8 stars of 21,476 above 1e-3.

In [ ]:
# Offline: just inspect the released scores and the fit flag.
shipped = np.load(assets.pls_encoding_path(), allow_pickle=False)
print('stars        :', shipped['pls'].shape)
print('in_age_fit   :', int(shipped['in_age_fit'].sum()), 'stars were used to fit')
print('score scale  : component-1 sd =', round(float(shipped['pls'][:, 0].std()), 2),
      '(raw, not z-scored)')

# To verify against your own latents:
#   uniq, Z = encotess.aggregate_by_star(latents_1536, gaia_ids, how='max')
#   T = pls.transform(Z)
#   # compare T to shipped['pls'] rows for the matching gaia_ids

### 3 · Keeping fewer components

Want the top 3 directions? Pass `dim=3` — components are nested, so that is exactly what a
3-component fit of this projection would give you. There is no separate artifact to load
and no refitting involved.

### Next

- **[2 · Encode a light curve](02_encode_a_lightcurve.ipynb)** — the PCA basis, the
  unsupervised counterpart to this projection.
- **[3 · Explore the released encodings](03_explore_the_encodings.ipynb)** — the latent
  space as a whole, with UMAP and nearest neighbours.